# 15 · 插件目录：安装成功之后还有哪些状态

安装、启用、可发现、schema 已加载、被允许调用，是五个不同判断。若只保留 installed 布尔值，停用或账户变化后，旧搜索结果仍可能被调用。
本章安装的是惰性 JSON 清单，不导入插件代码。实际写盘与异步装载让我们观察发布点、旧目录失效与取消后的恢复。
工具引用携带目录代次和内容摘要；await 返回后重新核对，才允许 schema 进入缓存。

**运行范围**：本章直接使用最终包中的共享组件，逐格验证本专题机制；不会启动模型、访问真实收费 API 或使用个人凭据。每次从第一格顺序运行，运行文件进入独立的 `runs/` 目录。改动参数后，从相应状态初始化格重新运行。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter
from unittest import TestCase
expect = TestCase()  # 只借用异常断言；下面逐格调用实现，不运行总测试替代实验。


## 源码事实与教学选择

固定对照提交：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。以下链接说明 Codex 对应机制；Python 类型和实验状态机是教学选择，运行它们不等于运行 Codex。

- [S05 · verified_plugin_install_completed](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/handlers/request_plugin_install.rs#L534-L547)：本地完成核验最终查相同 ID 的 plugin.installed，不在此要求 plugin.enabled。
- [S18 · ToolExposure](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/tools/src/tool_executor.rs#L49-L99)：ToolExposure 六种状态分别定义 initial/search/nested surface；Hidden 仍注册用于 dispatch，但不向模型显示。
- [S23 · handle_call/search_code_mode/search](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/handlers/tool_search.rs#L209-L349)：普通模型 search 返回 LoadableToolSpec；nested search 使用当前 step router 映射 canonical name、验证 normalization 后身份、取当前 runtime definition，再截断。


## 实验 1 · 安装与启用分开


构建本地清单，经过请求与确认写入 installed.json。默认不开启能力；重新读取已安装状态与查看有效目录是不同动作。
确认参数是实验宿主输入，不来自插件正文。


In [2]:
plugins = load_example('15', 'catalog')
import asyncio, json
run_dir = fresh_run_dir('ch15')
manifest = {'plugin_id':'task-tools', 'version':'1.0', 'environment_id':'local', 'tools':[
    {'name':'task.inspect','description':'inspect task status','exposure':'Deferred','schema':{'type':'object'}},
    {'name':'task.local','description':'local helper','exposure':'CodeModeOnly','schema':{'type':'object'}},
    {'name':'task.hidden','description':'internal helper','exposure':'Hidden','schema':{'type':'object'}}]}
source = run_dir/'manifest.json'
source.write_text(json.dumps(manifest))
store = plugins.PluginStore(run_dir/'installed')
catalog = plugins.ToolCatalog(store)
request = store.request_install(source, root_request=True, reason='Synthetic user requested task tools', candidates={'task-tools'})
assert await store.confirm_install(request, accepted=True) == 'installed'
assert 'task-tools' in store.state and catalog.entries() == {}
await store.enable('task-tools', True)
assert len(catalog.entries()) == 3
print({'installed': list(store.state), 'search_visible': [r.name for r,_ in catalog.list_exposed('search')]})


{'installed': ['task-tools'], 'search_visible': ['task.inspect']}


## 实验 2 · 搜索命中不等于允许调用


Deferred 工具先经搜索和 schema 加载，才进入模型调用面；CodeModeOnly 只进入 nested 面。
即使可见且 schema 已加载，执行策略仍可拒绝。试着改变 exposure 需要重新发布清单，不能只改旧引用。


In [3]:
ref = catalog.search('task status')[0]
with expect.assertRaisesRegex(plugins.Denied, 'unavailable'):
    catalog.resolve_call(ref, surface='model', authorize=lambda ref: True)
schema = await catalog.load_schema(ref)
assert schema == {'type':'object'}
assert catalog.resolve_call(ref, surface='model', authorize=lambda ref: True) == ref
with expect.assertRaisesRegex(plugins.Denied, 'policy denied'):
    catalog.resolve_call(ref, surface='model', authorize=lambda ref: False)
local_ref = catalog.entries()['task.local'][0]
assert catalog.resolve_call(local_ref, surface='nested', authorize=lambda ref: True) == local_ref
with expect.assertRaisesRegex(plugins.Denied, 'unavailable'):
    catalog.resolve_call(local_ref, surface='model', authorize=lambda ref: True)
assert 'task.hidden' not in [r.name for r,_ in catalog.list_exposed('nested')]
print({'schema': schema, 'index_builds': catalog.index_builds})


{'schema': {'type': 'object'}, 'index_builds': 1}


## 实验 3 · 认证变化跨过 await，禁止旧 schema 发布


装载 schema 期间变更认证代次，旧请求应在返回后被 publication fence 拦截。
缓存清空只处理已缓存数据；返回时重查才处理正在飞行的请求。


In [4]:
entered, release = asyncio.Event(), asyncio.Event()
catalog.schema_cache.clear()
async def delayed_schema(tool):
    entered.set()
    await release.wait()
    return tool['schema']
loading = asyncio.create_task(catalog.load_schema(ref, loader=delayed_schema))
await asyncio.wait_for(entered.wait(), 2)
store.auth_changed()
release.set()
with expect.assertRaisesRegex(plugins.StaleCatalog, 'generation changed'):
    await asyncio.wait_for(loading, 2)
assert ref not in catalog.schema_cache
new_ref = catalog.search('task status')[0]
assert new_ref.token != ref.token
await catalog.load_schema(new_ref)
await store.enable('task-tools', False)
with expect.assertRaises(plugins.StaleCatalog):
    catalog.resolve_call(new_ref, surface='model', authorize=lambda ref: True)
assert catalog.entries() == {}
print('在途 schema 被拒；停用后的旧引用不能调用。')


在途 schema 被拒；停用后的旧引用不能调用。


## 实验 4 · 提交前取消不留下半安装


stage 文件只是暂存，原子替换 installed.json 才是发布点。利用 barrier 在提交前取消，重开 store 验证状态。
这里不提供掉电持久性或多进程锁，只验证本地单进程事务式发布。


In [5]:
from copy import deepcopy
cancel_manifest = deepcopy(manifest)
cancel_manifest['plugin_id'] = 'cancelled-plugin'
cancel_source = run_dir/'cancelled.json'
cancel_source.write_text(json.dumps(cancel_manifest))
request = store.request_install(cancel_source, root_request=True, reason='Cancellation fixture', candidates={'cancelled-plugin'})
entered = asyncio.Event()
async def before_commit():
    entered.set()
    await asyncio.Event().wait()
installing = asyncio.create_task(store.confirm_install(request, accepted=True, before_commit=before_commit))
await asyncio.wait_for(entered.wait(), 2)
installing.cancel()
with expect.assertRaises(asyncio.CancelledError):
    await installing
reopened = plugins.PluginStore(run_dir/'installed')
assert 'cancelled-plugin' not in reopened.state
assert list((run_dir/'installed').glob('.stage-*')) == []
print({'after_cancel': list(reopened.state), 'staging_files': 0})


{'after_cancel': ['task-tools'], 'staging_files': 0}


## 实验 5 · 提交后丢回执，从持久状态恢复判断


把 `lose_ack=True` 放在发布之后：调用者收到连接异常，但磁盘已经安装。下一步应查询状态，
而不是把异常解释成“完全没发生”并盲目重做。进程重开生成新 epoch，旧 ToolRef 必须失效。


In [6]:
request = store.request_install(source, root_request=True, reason='Lost acknowledgement fixture', candidates={'task-tools'})
with expect.assertRaisesRegex(ConnectionError, 'committed; response lost'):
    await store.confirm_install(request, accepted=True, enabled=True, lose_ack=True)
recovered_store = plugins.PluginStore(run_dir/'installed')
recovered_catalog = plugins.ToolCatalog(recovered_store)
assert recovered_store.state['task-tools']['enabled']
assert recovered_store.token.epoch != store.token.epoch
with expect.assertRaises(plugins.StaleCatalog):
    await recovered_catalog.load_schema(new_ref)
fresh_ref = recovered_catalog.search('task status')[0]
assert await recovered_catalog.load_schema(fresh_ref) == {'type':'object'}
print({'disk_state': 'installed and enabled', 'fresh_epoch_required': True})


{'disk_state': 'installed and enabled', 'fresh_epoch_required': True}


## 你现在可以解释什么

能画出 installed → enabled → exposed → schema loaded → authorized 的不同检查，并说明为什么缓存失效与在途返回复查要同时存在。没有真实插件市场、OAuth、包代码执行或完整工具 Dispatcher。

**验证边界**：本章断言验证当前局部实现与合成输入。未运行固定提交的 Rust 测试，未验证真实模型、外部服务或累计 Agent 的端到端行为。
